In [2]:
import pandas as pd
import numpy as np

# 1. Configuración del Plan de Cuentas Corporativo (Para que SQL no sufra)
# Estructura: {Nombre: [Rango_Desde, Rango_Hasta, Tipo]}
plan_cuentas = {
    'Ventas Alojamiento':    [1000, 1099, 'Ingresos'],
    'Ventas F&B':            [1100, 1199, 'Ingresos'],
    'Otros Ingresos':        [1200, 1299, 'Ingresos'],
    'Costos Directos':       [2000, 2999, 'Costos'],
    'Gastos Operativos':     [3000, 3999, 'Gastos OP'],
    'Gastos No Operativos':  [4000, 4999, 'Gastos No OP']
}

# 2. Configuración de Países y Capacidad
paises = {
    'Argentina': {'moneda': 'ARS', 'habitaciones': 150, 'multiplicador': 1500},
    'Uruguay':   {'moneda': 'UYU', 'habitaciones': 80,  'multiplicador': 43},
    'Brasil':    {'moneda': 'BRL', 'habitaciones': 250, 'multiplicador': 6},
    'Mexico':    {'moneda': 'MXN', 'habitaciones': 300, 'multiplicador': 18}
}

fechas = pd.date_range('2025-01-01', '2025-12-31')
data_final = []

# 3. Generación con Lógica de Negocio
for pais, cfg in paises.items():
    for fecha in fechas:
        # A. GENERAR VENTAS (Basado en ocupación 65-85%)
        ocupacion = int(cfg['habitaciones'] * np.random.uniform(0.65, 0.85))
        for _ in range(ocupacion):
            # Venta de Habitación (Accomodation)
            tarifa_base_usd = np.random.uniform(100, 250)
            data_final.append({
                'Fecha': fecha, 'ID_Pais': pais, 'Moneda': cfg['moneda'],
                'Depto': 'Accomodation', 'Cuenta_Local': np.random.randint(1000, 1099),
                'Importe_Local': tarifa_base_usd * cfg['multiplicador']
            })
            # Venta F&B (Probabilidad de que el huésped consuma algo: 40%)
            if np.random.random() < 0.40:
                ticket_fb_usd = np.random.uniform(15, 45)
                data_final.append({
                    'Fecha': fecha, 'ID_Pais': pais, 'Moneda': cfg['moneda'],
                    'Depto': 'F&B', 'Cuenta_Local': np.random.randint(1100, 1199),
                    'Importe_Local': ticket_fb_usd * cfg['multiplicador']
                })
            # Venta Misc (Probabilidad de que el huésped consuma algo: 20%)
            if np.random.random() < 0.20:
                ticket_mi_usd = np.random.uniform(5, 25)
                data_final.append({
                    'Fecha': fecha, 'ID_Pais': pais, 'Moneda': cfg['moneda'],
                    'Depto': 'Miscellaneous', 'Cuenta_Local': np.random.randint(1200, 1299),
                    'Importe_Local': ticket_mi_usd * cfg['multiplicador']
                })

        # B. GENERAR COSTOS (Lógica por Departamento)
        margenes_costo = {
                'Accomodation': (0.15, 0.25),
                'F&B': (0.30, 0.40),
                'Miscellaneous': (0.10, 0.20)
                }

        for depto, (min_p, max_p) in margenes_costo.items():
            venta_actual = sum(item['Importe_Local'] for item in data_final
                       if item['Fecha'] == fecha and item['ID_Pais'] == pais and item['Depto'] == depto)
            if venta_actual > 0:
              porcentaje_random = np.random.uniform(min_p, max_p)
              costo_total_depto = (venta_actual * porcentaje_random) * -1
            data_final.append({
            'Fecha': fecha,
            'ID_Pais': pais,
            'Moneda': cfg['moneda'],
            'Depto': depto,
            'Cuenta_Local': np.random.randint(2000, 2999), # Rango de Costos Directos
            'Importe_Local': costo_total_depto
             })

        # C. GENERAR GASTOS DIARIOS (Lógica por Departamento)
        gastos_config = [
            ('Administracion', 3000, 3999, 2500, 12500),  # Gastos OP
            ('Mantenimiento',  3000, 3999, 1250, 7500),   # Gastos OP
            ('HR',             3000, 3999, 1000, 5000),   # Gastos OP
            ('Ventas',         3000, 3999, 750, 3750),   # Gastos OP
            ('Otros',          4000, 4999, 2000, 10000)  # Gastos No Operativos
        ]

        for depto, c_desde, c_hasta, v_min, v_max in gastos_config:
            monto_usd = np.random.uniform(v_min, v_max)
            data_final.append({
                'Fecha': fecha, 'ID_Pais': pais, 'Moneda': cfg['moneda'],
                'Depto': depto, 'Cuenta_Local': np.random.randint(c_desde, c_hasta),
                'Importe_Local': monto_usd * cfg['multiplicador']*-1
            })

df_final = pd.DataFrame(data_final)
df_final.to_csv('transacciones_contables_reales.csv', index=False)
print(f"✅ ¡Dataset robusto generado! Total registros: {len(df_final)}")

✅ ¡Dataset robusto generado! Total registros: 351902


In [3]:
# 1. Definimos los tipos de cambio base (promedios 2024 simulados)
# ARS: 1500, UYU: 43, BRL: 6, MXN: 18
tc_bases = {'ARS': 1500, 'UYU': 43, 'BRL': 6, 'MXN': 18}

data_tc = []

for fecha in fechas: # Usamos el mismo rango de fechas que el script de transacciones
    for moneda, base in tc_bases.items():
        # Simulamos una pequeña volatilidad diaria (0.5%) para que el reporte tenga "ruido" real
        variacion = np.random.uniform(-0.005, 0.005)
        tc_diario = round(base * (1 + variacion), 4)

        data_tc.append({
            'Fecha': fecha,
            'Moneda': moneda,
            'Tipo_Cambio_USD': tc_diario
        })

df_tc_robusto = pd.DataFrame(data_tc)
df_tc_robusto.to_csv('dimension_tc_robusta.csv', index=False)

print(f"✅ Tabla de TC generada con {len(df_tc_robusto)} registros.")

✅ Tabla de TC generada con 1460 registros.
